# 04 · Limpieza, imputación y transformación de Work-Olist

Auditoría de validez, unicidad y faltantes; criterios para tratar atípicos; imputación comparada; escalamiento, codificación, discretización, ingeniería de variables y trazabilidad.

**Modalidad:** práctica guiada, código ejecutable, interpretación y ejercicios. **Datos:** nueve CSV aportados por el usuario en `data/raw/`, con el esquema relacional de Olist. Se calculan resultados sobre estos archivos, sin simular registros ni cambiar las fuentes.

## Objetivos y decisiones de seguridad metodológica
Este notebook **no modifica la fuente**. Cada decisión se documenta y se compara antes/después. Enseña completitud, consistencia, unicidad, imputación simple y por grupo, MCAR/MAR/MNAR como hipótesis, IQR, sensibilidad, normalización, estandarización, `log1p`, `cut`, `qcut`, one-hot, indicadores, variables temporales y un pipeline mínimo.

**Criterios fundamentales:** NA ≠ cero; valor atípico ≠ error; media ≠ mediana; correlación ≠ causalidad. La elección de método debe ser apropiada a la variable y a la pregunta. La selección predictiva de características se menciona como extensión, **no es requisito del proyecto**.

**Origen:** CSV recibidos en `data/raw/`. Los valores faltantes, duplicados y extremos detectados se contabilizan sobre los archivos aportados. **No** se introdujeron anomalías artificiales.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
import numpy as np
import pandas as pd
# Detectar carpeta del proyecto desde raíz/ o notebooks/ (sin rutas absolutas).
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "olist_tools.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ejecute el notebook dentro de la carpeta Work_Olist_Notebooks_2026 (README.md).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.olist_tools import (read_tables, build_order_level, build_item_level,
                            table_inventory, missing_profile, safe_minmax, safe_zscore)
TABLAS, ORIGEN = read_tables(ROOT)
print("ORIGEN:", ORIGEN)
print("Directorio de trabajo:", ROOT)

### Diagnóstico real antes de intervenir

En los archivos aportados, `products` presenta **610 categorías ausentes** y únicamente **2 pesos faltantes**. Estos dos casos permiten contrastar estrategias de imputación, pero una variación agregada muy pequeña **no demuestra** que la estrategia sea universalmente adecuada. No se inventan observaciones faltantes artificiales para justificar una técnica.

En `geolocation` se observan **261 831 filas exactamente repetidas**. Esto no basta para declararlas errores: la tabla no declara una llave única por coordenada y puede contener repeticiones legítimas. Antes de agregar o eliminar, definimos el objetivo y la unidad de observación. Del mismo modo, la presencia de varios pagos o reseñas por pedido no es un duplicado por sí misma.


In [ ]:
from datetime import datetime
from IPython.display import display
orders_raw=build_order_level(TABLAS)
items_raw=build_item_level(TABLAS)
registro=[]
def log_decision(paso,tabla,variable,criterio,afectadas,accion,limitacion):
    registro.append({'paso':paso,'tabla':tabla,'variable':variable,'criterio':criterio,
                     'filas_afectadas':int(afectadas),'accion':accion,'limitacion':limitacion})
print('Origen:',ORIGEN,'| pedidos:',len(orders_raw),'| ítems:',len(items_raw))

## 1. Auditoría de tipos, rangos, llaves y reglas temporales
**Diagnóstico antes que corrección.** Los identificadores tienen que respetar la cardinalidad. Para los pedidos entregados, la fecha de entrega no debería preceder la compra; para los demás, una fecha vacía puede ser estructural. El precio y el flete no deberían ser negativos, pero no declararemos que todas las filas extremas son errores.

In [ ]:
checks=pd.DataFrame([
 {'control':'order_id único','incumplen':int(orders_raw['order_id'].duplicated().sum())},
 {'control':'(order_id, order_item_id) único','incumplen':int(TABLAS['items'].duplicated(['order_id','order_item_id']).sum())},
 {'control':'precio < 0','incumplen':int(TABLAS['items']['price'].lt(0).sum())},
 {'control':'flete < 0','incumplen':int(TABLAS['items']['freight_value'].lt(0).sum())},
 {'control':'días hasta entrega < 0','incumplen':int(orders_raw['dias_hasta_entrega'].lt(0).sum())},
 {'control':'reseña fuera de 1 a 5','incumplen':int((TABLAS['reviews']['review_score'].notna() & ~TABLAS['reviews']['review_score'].between(1,5)).sum())},
])
display(checks)
for _,row in checks.iterrows():
    log_decision('diagnostico','varias',row['control'],'regla de consistencia',row['incumplen'],
                 'solo marcar; aún no corregir','requiere revisar procedencia')

## 2. Magnitud y mecanismo de valores faltantes
Se distingue MCAR (falta completamente al azar), MAR (depende de variables observadas) y MNAR (depende del valor no observado). **Ningún mecanismo puede confirmarse simplemente mirando una tabla de NA**. La ausencia de entrega en un pedido cancelado se debe tratar como *no aplicable*, no como cero días ni como caso para imputar una duración.

In [ ]:
display(missing_profile(orders_raw).head(16))
print('Proporción con fecha de entrega faltante por estatus:')
display(orders_raw.groupby('order_status')['order_delivered_customer_date'].apply(lambda s:s.isna().mean()).round(3).to_frame('fraccion_na'))

## 3. Plan de tratamiento (conservador)
La pérdida de casos por `dropna` se compara antes de utilizarla. Elegiremos una variable física relativamente apropiada para una **demostración de imputación**: `product_weight_g` de la tabla de productos. Un peso ausente puede imputarse como escenario didáctico; el precio de un pedido o la fecha de entrega **no** se inventan automáticamente porque tienen consecuencias operativas.

In [ ]:
productos=TABLAS['products'].copy()
peso=pd.to_numeric(productos['product_weight_g'],errors='coerce')
print('Productos:',len(productos),'| peso faltante:',int(peso.isna().sum()))
lista_completa=productos.dropna(subset=['product_weight_g','product_category_name'])
print('Productos completos para peso + categoría:',len(lista_completa),
      '| descartados:',len(productos)-len(lista_completa))
log_decision('comparacion','products','product_weight_g','complete case',len(productos)-len(lista_completa),
             'comparar sin aplicar eliminación a la tabla fuente','puede cambiar distribución y reducir n')

## 4. Imputación: mediana global, por categoría e indicador de ausencia
La imputación simple preserva filas, pero **reduce artificialmente la variabilidad** y puede distorsionar asociaciones. Imputar por grupo requiere que la categoría esté disponible y que existan suficientes observaciones. Añadimos un indicador binario de faltante y comparamos las alternativas; el original permanece intacto.

In [ ]:
productos_trabajo=productos.copy()
productos_trabajo['peso_faltaba']=productos_trabajo['product_weight_g'].isna()
productos_trabajo['peso_global']=peso.fillna(peso.median())
mediana_categoria=productos_trabajo.groupby('product_category_name')['product_weight_g'].transform('median')
productos_trabajo['peso_por_categoria']=peso.fillna(mediana_categoria).fillna(peso.median())
comparativa=pd.DataFrame({
 'escenario':['original observado','mediana global','mediana por categoría'],
 'n':[peso.notna().sum(),productos_trabajo['peso_global'].notna().sum(),productos_trabajo['peso_por_categoria'].notna().sum()],
 'media':[peso.mean(),productos_trabajo['peso_global'].mean(),productos_trabajo['peso_por_categoria'].mean()],
 'desv_std':[peso.std(),productos_trabajo['peso_global'].std(),productos_trabajo['peso_por_categoria'].std()]})
display(comparativa.round(2))
log_decision('imputacion','products','product_weight_g','mediana global y condicional por categoría',
             productos_trabajo['peso_faltaba'].sum(),'generar columnas alternativas, no sobrescribir',
             'imputación simple no conserva la varianza ni la incertidumbre')

In [ ]:
# Comparar representaciones ANTES y DESPUÉS de imputar
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,4))
valor_max=productos_trabajo['peso_por_categoria'].quantile(.98)
for s,label,color in [(peso,'Original','#58c6f5'),(productos_trabajo['peso_por_categoria'],'Por categoría','#ef8bd8')]:
    ax.hist(s.dropna().clip(upper=valor_max),bins=22,alpha=.48,label=label,color=color)
ax.set(title='Peso de producto: sensibilidad a la imputación',xlabel='Peso en gramos (vista truncada a P98)',ylabel='Frecuencia')
ax.legend();plt.tight_layout();plt.show();plt.close(fig)

### Imputación de categóricas: ¿moda o categoría explícita?
Para una categoría sin registro, sustituir por moda puede crear falsa concentración. Un nivel «Sin información» preserva su condición de ausencia, aunque no reconstruye la verdadera categoría. El uso depende de la finalidad; nunca confundir «no registrado» con una categoría comercial real.

In [ ]:
productos_trabajo['categoria_faltaba']=productos_trabajo['product_category_name'].isna()
productos_trabajo['categoria_informada']=productos_trabajo['product_category_name'].fillna('SIN_INFORMACION')
print(productos_trabajo[['categoria_faltaba','categoria_informada']].head())
log_decision('imputacion_categorica','products','product_category_name','NA explícito',
             productos_trabajo['categoria_faltaba'].sum(),'crear categoría SIN_INFORMACION',
             'no sustituye etiqueta verdadera')

## 5. Errores confirmables vs atípicos plausibles
Una cantidad negativa en precio o flete contradice la regla simple de no negatividad y debe **marcarse**, pero si el sistema registra devoluciones con signo, la regla cambia. Un precio muy alto puede ser legítimo. Para detectar extremos calcularemos IQR y compararemos una transformación `clip` **solo como escenario de sensibilidad**, sin reemplazar automáticamente el original.

In [ ]:
it=items_raw.copy()
it['precio_invalido']=it['price'].lt(0)
it['flete_invalido']=it['freight_value'].lt(0)
it['precio_revisado']=it['price'].mask(it['precio_invalido'])  # error sospechoso -> NA pendiente de investigación
it['flete_revisado']=it['freight_value'].mask(it['flete_invalido'])
s=it['precio_revisado'].dropna(); q1,q3=s.quantile([.25,.75]);ric=q3-q1
lo,hi=q1-1.5*ric,q3+1.5*ric
it['precio_atipico_IQR']=it['precio_revisado'].lt(lo)|it['precio_revisado'].gt(hi)
print('Límites IQR:',round(lo,2),round(hi,2),'| marcados:',int(it['precio_atipico_IQR'].sum()))
print('Negativos sospechosos:',int(it['precio_invalido'].sum()))
log_decision('validez','items','price/freight_value','valores negativos',
             it['precio_invalido'].sum()+it['flete_invalido'].sum(),
             'marcar; convertir a NA SOLO en columnas de trabajo',
             'confirmar si se representan devoluciones o errores')
log_decision('atipicos','items','price','1.5 × RIC',it['precio_atipico_IQR'].sum(),
             'marcar únicamente','asimetría puede producir extremos legítimos')

In [ ]:
# Sensibilidad: recorte opcional de extremos y sus efectos estadísticos.
it['precio_capado_ESCENARIO']=it['precio_revisado'].clip(lower=max(0,lo),upper=hi)
display(pd.DataFrame({
    'escenario':['sin recorte','escenario de recorte IQR'],
    'media':[it['precio_revisado'].mean(),it['precio_capado_ESCENARIO'].mean()],
    'mediana':[it['precio_revisado'].median(),it['precio_capado_ESCENARIO'].median()],
    'desv_std':[it['precio_revisado'].std(),it['precio_capado_ESCENARIO'].std()] }).round(2))
print('ATENCIÓN: este recorte NO se aplica a los pedidos originales.')

## 6. Fechas: inconsistencias y ausencia estructural
No imputamos fechas de entrega para pedidos cancelados, creados o no entregados. Para entregados con fecha negativa, conservamos la fila, marcamos la duración como NA en una columna derivada **hasta revisar el registro**, y creamos un indicador. Esto distingue limpieza reproducible de modificación opaca.

In [ ]:
ord_t=orders_raw.copy()
ord_t['duracion_negativa']=ord_t['dias_hasta_entrega'].lt(0)
ord_t['dias_entrega_validada']=ord_t['dias_hasta_entrega'].mask(ord_t['duracion_negativa'])
print('Duraciones negativas marcadas:',int(ord_t['duracion_negativa'].sum()))
print('Cancelados con duración NA (no se imputa):',
      int(ord_t.loc[ord_t.order_status.eq('canceled'),'dias_entrega_validada'].isna().sum()))
log_decision('consistencia_temporal','orders','dias_hasta_entrega','duración negativa',
             ord_t['duracion_negativa'].sum(),'marcar NA en una copia','no inventar fecha de entrega')

## 7. Escalamiento, estandarización y log-transformación
Min–max: $x^*=(x-\min x)/(\max x-\min x)$; z-score: $z=(x-\bar{x})/s$. Son operaciones diferentes. Una variable constante tiene denominador cero y requiere decisión documentada. `log1p` necesita entrada mayor o igual a −1; aquí se usa en precios **no negativos**. Estos escaladores son descriptivos; en un problema predictivo se ajustarían solamente con datos de entrenamiento, evitando *data leakage*.

In [ ]:
precio_no_neg=it['precio_revisado'].where(it['precio_revisado'].ge(0))
it['precio_minmax']=safe_minmax(precio_no_neg)
it['precio_zscore']=safe_zscore(precio_no_neg)
it['precio_log1p']=np.log1p(precio_no_neg)
prueba_constante=pd.Series([7.0,7.0,7.0])
print('Constante min-max:',safe_minmax(prueba_constante).to_list())
print('Constante z-score:',safe_zscore(prueba_constante).to_list())
display(it[['price','precio_revisado','precio_minmax','precio_zscore','precio_log1p']].head(9))
log_decision('transformacion','items','price','min-max, z, log1p',precio_no_neg.notna().sum(),
             'crear columnas nuevas','log requiere valores no negativos; escalar cambia unidades')

## 8. Discretización y variables indicadoras
`pd.cut` usa puntos de corte substantivos; `pd.qcut` asigna por cuantiles muestrales (con `duplicates='drop'` por valores repetidos). Codificación one-hot se usa para nominales; números enteros arbitrarios para categorías nominales crean un orden espurio. Las reseñas 1–5 ya poseen orden, pero no necesariamente intervalos equidistantes.

In [ ]:
ord_t['tramo_valor_BRL']=pd.cut(ord_t['subtotal_productos'],
 bins=[-np.inf,50,150,400,np.inf],labels=['≤50','50–150','150–400','>400'])
ord_t['cuartil_valor']=pd.qcut(ord_t['subtotal_productos'],q=4,labels=False,duplicates='drop')
dummies=pd.get_dummies(ord_t['medio_pago_inicial'].fillna('SIN_INFORMACION'),
 prefix='pago',dtype='int8')
print('Columnas one-hot:',dummies.columns.to_list())
display(ord_t[['subtotal_productos','tramo_valor_BRL','cuartil_valor']].head())
print('Una fila nominal no debe confundirse con una categoría ordinal numerada sin justificación.')
log_decision('codificacion','orders','medio_pago_inicial','one-hot',len(ord_t),
             'generar variables indicadoras','no inventar ordinalidad')

## 9. Ingeniería de variables con significado claro
Creamos duración validada, diferencia frente a fecha estimada, mes de compra e indicador de atraso **solamente cuando hay fechas disponibles**. Una razón flete/precio requiere denominador estrictamente positivo. Este proceso es descriptivo, no una selección automática para modelos predictivos.

In [ ]:
ord_t['proporcion_flete']=np.where(ord_t['subtotal_productos'].gt(0),
  ord_t['total_flete']/ord_t['subtotal_productos'],np.nan)
ord_t['mes_compra']=ord_t['order_purchase_timestamp'].dt.to_period('M').astype('string')
ord_t['llegada_tardia_validada']=(ord_t['dias_retraso'].gt(0) & ord_t['dias_entrega_validada'].notna()).astype('boolean')
ord_t.loc[ord_t['dias_entrega_validada'].isna(),'llegada_tardia_validada']=pd.NA
cols=['mes_compra','dias_entrega_validada','llegada_tardia_validada','proporcion_flete','tramo_valor_BRL']
display(ord_t[cols].head())

## 10. Pipeline mínimo: original → diagnóstico → versiones
En vez de aplicar modificaciones irreversibles, encapsulamos pasos en una función que devuelve copia, y registramos cada modificación. Se pueden construir pipelines con `sklearn`, pero aquí una función clara es suficiente. **Si se tratara de predicción, cualquier imputador o escalador debería ajustarse SOLO sobre entrenamiento**.

In [ ]:
def preparar_pedidos_exploratorios(df: pd.DataFrame) -> pd.DataFrame:
    """Crea variables derivadas conservando columnas de origen y el número de pedidos."""
    nuevo=df.copy()
    nuevo['duracion_negativa']=nuevo['dias_hasta_entrega'].lt(0)
    nuevo['dias_entrega_validada']=nuevo['dias_hasta_entrega'].mask(nuevo['duracion_negativa'])
    nuevo['valor_log1p']=np.log1p(nuevo['subtotal_productos'].where(nuevo['subtotal_productos'].ge(0)))
    nuevo['valor_z']=safe_zscore(nuevo['subtotal_productos'])
    nuevo['valor_minmax']=safe_minmax(nuevo['subtotal_productos'])
    return nuevo
resultado=preparar_pedidos_exploratorios(orders_raw)
assert len(resultado)==len(orders_raw) and resultado['order_id'].is_unique
assert orders_raw['order_id'].equals(resultado['order_id'])
print('Conservadas',len(resultado),'filas y llaves originales.')

## 11. Bitácora, exportación y sensibilidad
Guardaremos una tabla de decisiones y una copia derivada, **nunca sobrescribimos `data/raw`**. Las imputaciones alternativas se conservan como columnas con nombres explícitos. En un trabajo real añadiríamos fecha, autor, versión del código, fuente y justificación de cada corrección concreta. Compara resúmenes y figuras antes/después: si cambia el resultado, documéntalo.

In [ ]:
bitacora=pd.DataFrame(registro)
display(bitacora)
out=ROOT/'data'/'processed';out.mkdir(parents=True,exist_ok=True)
resultado.to_csv(out/'olist_pedidos_preparados.csv',index=False)
productos_trabajo.to_csv(out/'olist_productos_imputacion_escenarios.csv',index=False)
bitacora.to_csv(out/'bitacora_limpieza.csv',index=False)
(out/'origen_limpieza.txt').write_text('Origen: '+ORIGEN+'\nArchivo bruto sin modificar; ejemplos de tratamiento alternativo.\n',encoding='utf8')
print('Guardado en data/processed/: versiones derivadas y bitácora.')

### Decisiones documentadas con datos reales

La **no negatividad** y el orden de los eventos temporales pueden verificarse en los registros; los **precios extremos** solo se marcan como candidatos a investigación. El escenario de recorte no es una política automática. Las imputaciones de `product_weight_g` se guardan en columnas alternativas, se incluye un indicador de faltante y se preserva el archivo original.

**Extensión optativa:** `KNNImputer`, imputación iterativa y selección de características requieren hipótesis, separación de entrenamiento/prueba si hay predicción y evaluación de sensibilidad; no constituyen exigencia obligatoria del proyecto exploratorio.


## 12. Panorama opcional: métodos avanzados y fuga de información
El material histórico menciona KNN, imputación iterativa y selección de características por filtro, envolvente y métodos embebidos. **Son ampliaciones**, no requisito esencial. Sin objetivos predictivos, particiones de entrenamiento ni evaluación, sería metodológicamente incorrecto presentar el resultado de una imputación compleja como «mejor» automáticamente. Si se exploran modelos, deben documentarse el ajuste en entrenamiento, las métricas y la sensibilidad.

### Ejercicios
- **Esencial.** Elabora una tabla de calidad por variable y documenta una intervención justificable.
- **Interpretación.** Compara mediana global y mediana por categoría para `product_weight_g`. ¿Qué ocurre con media y desviación?
- **Extensión.** Compara `precio_revisado` con el escenario de recorte: interpreta por qué la media cambia más que la mediana. Propón una política de no eliminación automática de extremos.

**Referencias:** presentación histórica de limpieza y transformación (faltantes, coherencia, unicidad, validez, normalización, discretización, codificación, nuevas variables, fuga de datos); matriz académica aprobada; [pandas missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html), [van Buuren, *Flexible Imputation of Missing Data*](https://stefvanbuuren.name/fimd/).

In [ ]:
# Zona de ejercicios: escriba la bitácora de sus decisiones y compare escenarios.